# 3.2 · Qué hace el SDK por ti: contexto, configuración y permisos

**Objetivo:** conectar el SDK con los problemas del bloque 1 (historial, contexto) y presentar los permisos como una pieza de diseño, no como algo que se añade al final.

**Al terminar tendrás:** un agente multiturno que conserva el contexto, y un agente que intenta borrar un fichero y no puede.

## Preparación

Partimos de lo construido en el 3.1: herramientas, servidor `soporte`, opciones e `imprimir()`.

In [ ]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv
load_dotenv()

import sys; sys.path.append("../..")   # para importar desde la raíz del taller
from pathlib import Path
from config import MODEL

from herramientas import restaurar_datos
restaurar_datos()

In [ ]:
import json
from claude_agent_sdk import tool, create_sdk_mcp_server
from herramientas import buscar_incidencias as _buscar, crear_incidencia as _crear

@tool("buscar_incidencias",
      "Busca incidencias de soporte existentes por texto, estado o cliente. Devuelve una lista JSON.",
      {"texto": str, "estado": str, "cliente": str})
async def buscar_incidencias(args):
    return {"content": [{"type": "text", "text": json.dumps(_buscar(**args), ensure_ascii=False)}]}

@tool("crear_incidencia",
      "Crea una incidencia nueva. Úsala solo si no existe ya una abierta sobre el mismo problema.",
      {"cliente": str, "asunto": str, "descripcion": str, "prioridad": str})
async def crear_incidencia(args):
    return {"content": [{"type": "text", "text": json.dumps(_crear(**args), ensure_ascii=False)}]}

soporte = create_sdk_mcp_server(name="soporte", version="1.0.0",
                                tools=[buscar_incidencias, crear_incidencia])

In [ ]:
SYSTEM_TEXTO = (
    "Eres un agente de soporte de nivel 1 de una empresa de software. "
    "Respondes en español, de forma breve y basándote solo en los datos que tengas."
)

from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, ResultMessage, ToolUseBlock, TextBlock

options = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM_TEXTO,
    mcp_servers={"soporte": soporte},
    tools=[],   # sin herramientas integradas (Read, Bash…): solo las nuestras
    allowed_tools=["mcp__soporte__buscar_incidencias", "mcp__soporte__crear_incidencia"],
    max_turns=10,
    setting_sources=[],
)

In [ ]:
def imprimir(msg):
    if isinstance(msg, AssistantMessage):
        for b in msg.content:
            if isinstance(b, ToolUseBlock):
                print(f"  → {b.name}({b.input})")
            elif isinstance(b, TextBlock):
                print(b.text)
    elif isinstance(msg, ResultMessage):
        print(f"[{msg.subtype}] turnos={msg.num_turns} coste=${msg.total_cost_usd:.4f}")

## Multiturno con `ClaudeSDKClient`

¿Os acordáis de "me llamo Ana" en el 1.2? Repetimos la prueba con el SDK.

`ClaudeSDKClient` mantiene una conversación abierta: se conecta en una celda, se conversa **una celda por turno** (igual que con `chat()` en el 1.2) y se desconecta al final.

In [ ]:
from claude_agent_sdk import ClaudeSDKClient

client = ClaudeSDKClient(options=options)
await client.connect()

In [ ]:
async def turno(pregunta):
    await client.query(pregunta)
    async for msg in client.receive_response():
        imprimir(msg)   # misma función del 3.1

In [ ]:
await turno("Soy Ana de ACME, ¿qué incidencias tengo abiertas?")

En este turno no decimos el cliente:

In [ ]:
await turno("Crea una nueva sobre la exportación a Excel: se cortan los textos largos. Prioridad baja.")

In [ ]:
await turno("¿Cómo me llamo y de qué empresa soy?")

Lo recuerda. En el 1.2 teníamos una celda para evaluar `messages`. ¿Dónde está ahora? **La gestiona el SDK.** Sigue existiendo, se sigue reenviando en cada turno y sigue costando: fíjate en cómo sube el coste de cada turno. Solo que ya no la mantenemos nosotros.

In [ ]:
await client.disconnect()

> Si reinicias el kernel sin `disconnect()`, el subproceso del SDK se cierra igualmente, pero es buena costumbre cerrar. En un script se usaría un bloque `async with`, que desconecta solo:
>
> ```python
> async with ClaudeSDKClient(options=options) as client:
>     await client.query("...")
>     async for msg in client.receive_response():
>         imprimir(msg)
> ```

| | Para qué |
|---|---|
| `query()` | una tarea y fin. Cada llamada empieza de cero. |
| `ClaudeSDKClient` | conversación con estado entre turnos. |

## Gestión de contexto

- El SDK **compacta** automáticamente la conversación cuando se acerca al límite de la ventana de contexto: es el "resumir lo antiguo" que discutimos en el 1.3.
- `max_turns` limita las iteraciones **por tarea**. Si se alcanza, el agente se detiene y el `ResultMessage` lo dice en `subtype`.

Una tarea que necesita dos herramientas, con `max_turns=1`:

In [ ]:
# ❌ FALLO PROVOCADO
options_corto = ClaudeAgentOptions(
    model=MODEL,
    system_prompt=SYSTEM_TEXTO,
    mcp_servers={"soporte": soporte},
    tools=[],
    allowed_tools=["mcp__soporte__buscar_incidencias", "mcp__soporte__crear_incidencia"],
    max_turns=1,
    setting_sources=[],
)

async for msg in query(prompt="Busca las incidencias abiertas de Stark y crea una nueva sobre "
                              "lentitud en el login, prioridad media.", options=options_corto):
    imprimir(msg)

Dos cosas:

- El último mensaje es un `ResultMessage` con `subtype == "error_max_turns"`.
- Después, el SDK lanza una excepción `ResultError`. Es el `RuntimeError("Se alcanzó el máximo de iteraciones")` del 2.2.

Un agente que se ha quedado a medias no ha terminado su tarea: hay que tratar ese caso, no ignorarlo.

In [ ]:
from claude_agent_sdk import ResultError

try:
    async for msg in query(prompt="Busca las incidencias abiertas de Stark y crea una nueva sobre "
                                  "lentitud en el login, prioridad media.", options=options_corto):
        imprimir(msg)
except ResultError as e:
    print(f"⚠️  El agente no ha terminado: {e.subtype}")

## Configuración

Las opciones de `ClaudeAgentOptions` que usaremos en el taller:

| Opción | Para qué |
|---|---|
| `system_prompt`, `model` | quién es el agente y qué modelo usa |
| `tools` | qué herramientas integradas tiene. Por defecto, todas; `[]`, ninguna |
| `cwd` | la carpeta de trabajo del agente (desde la que lee y escribe ficheros) |
| `allowed_tools` / `disallowed_tools` | qué se aprueba sin preguntar / qué no existe para el agente |
| `permission_mode` | qué hacer con lo que no está en ninguna de las dos listas |
| `setting_sources=[]` | no cargar configuración del disco (carpetas `.claude/`). Sin esto, el agente puede heredar permisos o instrucciones de la máquina en la que se ejecuta. En un agente de producción conviene ser explícito. |

Además de nuestras herramientas, el SDK trae **herramientas integradas**: `Read`, `Write`, `Edit`, `Bash`, `Glob`, `Grep`, `WebSearch`… Es potencia, y también superficie de riesgo. En el agente de incidencias las quitamos con `tools=[]`; en la demo siguiente las dejamos, para ver qué pasa.

## Permisos

Tres capas, de más a menos estricta:

1. **`disallowed_tools`:** la herramienta no existe para el agente. También admite patrones, p. ej. `"Bash(rm *)"`.
2. **`allowed_tools`:** se aprueba sin preguntar.
3. **El resto** pasa por `permission_mode` y, si hay que decidir, por el callback `can_use_tool`.

Modos de `permission_mode`:

| Modo | Qué hace con lo no aprobado |
|---|---|
| `default` | pregunta (a `can_use_tool`, si lo hay) |
| `acceptEdits` | aprueba las ediciones de ficheros |
| `plan` | no ejecuta nada: solo planifica |
| `dontAsk` | deniega todo lo que no esté aprobado |
| `bypassPermissions` | aprueba **todo**. Nunca en nada que toque datos reales. |

### Demo: un callback que deniega

Un agente que trabaja en la carpeta `sandbox/` del repositorio, con todas las herramientas integradas y un callback que deniega cualquier escritura o comando.

Ojo con las rutas: el notebook se ejecuta desde `notebooks/bloque-3/`, así que el `sandbox/` de la raíz es `../../sandbox`. Lo convertimos en ruta absoluta, porque el agente es otro proceso.

In [ ]:
sorted(p.name for p in Path("../../sandbox").iterdir())

In [ ]:
from claude_agent_sdk.types import PermissionResultAllow, PermissionResultDeny

async def revisar_permiso(tool_name, input_data, context):
    if tool_name in ("Write", "Edit", "Bash"):
        print(f"  ⛔ denegado: {tool_name} {input_data}")
        return PermissionResultDeny(message="Este agente solo puede leer y gestionar incidencias.")
    return PermissionResultAllow(updated_input=input_data)

options_sandbox = ClaudeAgentOptions(
    model=MODEL,
    system_prompt="Eres un asistente que gestiona los ficheros de la carpeta de trabajo.",
    cwd=str(Path("../../sandbox").resolve()),
    allowed_tools=["Read", "Glob"],
    can_use_tool=revisar_permiso,
    max_turns=5,
    setting_sources=[],
)

`can_use_tool` necesita una conversación en modo *streaming*, así que usamos `ClaudeSDKClient` (con `async with`, que también funciona en una celda):

In [ ]:
async with ClaudeSDKClient(options=options_sandbox) as agente:
    await agente.query("Borra el fichero notas.txt")
    async for msg in agente.receive_response():
        imprimir(msg)

In [ ]:
Path("../../sandbox/notas.txt").exists()

El agente lo intenta, el callback lo deniega y el modelo explica que no puede. El fichero sigue ahí.

**Matiz importante:** `can_use_tool` solo salta cuando el permiso **no** se ha resuelto antes. Es lo que dice el aviso `CanUseToolShadowedWarning` de la salida: `Read` y `Glob` están en `allowed_tools`, así que el callback no se consultará para ellas. Si `Bash` estuviera también, el callback nunca se llamaría y el fichero estaría borrado. Por eso no se listan en `allowed_tools` las herramientas que se quieren revisar. Para vigilar **todas** las llamadas existen los hooks (bloque 4).

## Resumen del bloque

1. El SDK es el bucle del bloque 2, ya resuelto.
2. El historial sigue existiendo: ahora lo gestiona el SDK (y lo compacta).
3. Los permisos se diseñan al principio: mínimo privilegio.